[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ry4n-felinto/impytech_2026/blob/dev/analise_impytech.ipynb)

In [ ]:
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt

df = pd.read_csv("avaliacao_impytech.csv")

# RENOMEAR COLUNAS
novos_nomes = {
    "Carimbo de data/hora": "data_hora",
    "Insira seu nome para contabilizar a presença na aula": "nome",
    "Qual aula você irá avaliar?": "aula",
    "Como você classifica o quanto você aprendeu sobre o conteúdo?": "aprendizado_sobre_conteudo",
    "Como você classifica o quanto você compreendeu sobre o conteúdo?": "compreensao_sobre_conteudo",
    "Como você classifica a performance do professor desta aula?": "performance_professor",
    "Como você classifica a performance dos monitores nessa aula?": "performance_monitores",
    "Você acredita que, o conteúdo que foi apresentado nessa aula pode lhe ajudar com atividades futuras?": "utilidade",
    "Existe algum tópico sobre a aula que você acredita que não tenha ficado claro?": "topicos_nao_claros",
    "Caso haja algum comentário e/ou sugestão sobre a aula, escreva aqui": "comentarios_aula",
    "Caso tenha algum ponto a destacar sobre o professor, escreva aqui": "comentarios_professor",
    "Caso tenha algum ponto a destacar sobre os monitores, escreva aqui": "comentarios_monitores",
    "Dê uma nota final para essa aula": "nota_final"
}

df.rename(columns=novos_nomes, inplace=True)

# REMOVER COLUNAS INÚTEIS

colunas_importantes = [
    "nome",
    "aula",
    "aprendizado_sobre_conteudo",
    "compreensao_sobre_conteudo",
    "performance_professor",
    "performance_monitores",
    "utilidade",
    "nota_final"
]

df = df[colunas_importantes].copy()

# OTIMIZAR COLUNA "AULA" PRA PEGAR SÓ O NÚMERO
df["aula"] = df["aula"].str[:6]

# OTIMIZAR COLUNA "COMPREENSAO" PRA PEGAR SÓ A NOTA
df["compreensao_sobre_conteudo"] = df["compreensao_sobre_conteudo"].str[0]

# CONSERTAR dtype
df = df.astype({
    "compreensao_sobre_conteudo": "int64",
})

display(df)

# **VISÃO GERAL**

In [ ]:
print(f"Número de respostas:\t\t{len(df)}")
print(f"Número de aulas:\t\t{df['aula'].nunique()}")
print(f"Nota média final:\t\t{df['nota_final'].mean():.2f}")
print(f"Aprendizado médio:\t\t{df['aprendizado_sobre_conteudo'].mean():.2f}")
print(f"Compreensão média:\t\t{df['compreensao_sobre_conteudo'].mean():.2f}")
print(f"Professor:\t\t\t{df['performance_professor'].mean():.2f}")
print(f"Monitores:\t\t\t{df['performance_monitores'].mean():.2f}")

## **1. Estatísticas principais**

Calcula as seguintes estatísticas:
- Média
- Mediana
- Moda
- Mínimo e máximo
- Variância

Para as colunas:
- Aprendizado e compreensão de conteúdo
- Performance dos professores e monitores
- Nota final

In [ ]:
colunas = [
    "aprendizado_sobre_conteudo",
    "compreensao_sobre_conteudo",
    "performance_professor",
    "performance_monitores",
    "nota_final"
]

for coluna in colunas:
    media = df[coluna].mean()
    mediana = df[coluna].median()
    moda = df[coluna].mode().tolist()
    minimo = df[coluna].min()
    maximo = df[coluna].max()
    variancia = df[coluna].var()

    print("=" * 50)
    print(f"ESTATÍSTICAS DE {coluna.upper()}")
    print("=" * 50)
    print(f"Número de avaliações:\t{len(df)}")
    print(f"Média:\t\t\t{media:.2f}")
    print(f"Mediana:\t\t{mediana:.2f}")
    print(f"Moda:\t\t\t{', '.join(map(str, moda))}")
    print(f"Mínimo:\t\t\t{minimo}")
    print(f"Máximo:\t\t\t{maximo}")
    print(f"Variância:\t\t{variancia:.2f}")
    print()

## **2. Médias por aula**

In [ ]:
media_por_aula = (
    df.groupby("aula")
      .mean(numeric_only=True)
      .round(2)
)

display(media_por_aula)

## **3. Ranking das aulas por média de nota final**

In [ ]:
ranking = (
    df.groupby("aula")["nota_final"]
      .mean()
      .sort_values(ascending=False)
      .round(2)
)

print("Ranking das aulas:")
display(ranking)

## **4. Critérios com maior média**

In [ ]:
medias = df[[
    "aprendizado_sobre_conteudo",
    "compreensao_sobre_conteudo",
    "performance_professor",
    "performance_monitores",
    "nota_final"
]].mean().sort_values(ascending=False)

display(medias.round(2))

## **5. Utilidade do conteúdo**

In [ ]:
display(df["utilidade"].value_counts())

# **GRÁFICOS**

In [ ]:
import altair as alt
import pandas as pd

def grafico_media_por_aula(df, coluna, titulo, dominio):
    # MÉDIA POR AULA
    media = (
        df.groupby("aula", as_index=False)[coluna]
        .mean()
    )

    # MÉDIA GERAL
    media_geral = media[coluna].mean()

    # GRÁFICO DE LINHA
    grafico = (
        alt.Chart(media)
        .mark_line(
            color="steelblue",
            strokeWidth=3,
            point=alt.OverlayMarkDef(
                filled=True,
                fill="orange",
                stroke="black",
                strokeWidth=1,
                size=130
            )
        )
        .encode(
            x=alt.X(
                "aula:N",
                title="Aula",
                sort=None
            ),
            y=alt.Y(
                f"{coluna}:Q",
                title="Média",
                scale=alt.Scale(domain=dominio)
            ),
            tooltip=[
                alt.Tooltip("aula:N", title="Aula"),
                alt.Tooltip(
                    f"{coluna}:Q",
                    title="Média",
                    format=".2f"
                )
            ]
        )
    )

    # LINHA NA MÉDIA
    linha_media = (
        alt.Chart(
            pd.DataFrame({"media": [media_geral]})
        )
        .mark_rule(
            color="red",
            strokeDash=[8, 6],
            size=2
        )
        .encode(
            y="media:Q"
        )
    )

    # TEXTO DE MÉDIA
    texto = (
        alt.Chart(
            pd.DataFrame({"media": [media_geral]})
        )
        .mark_text(
            color="red",
            align="left",
            baseline="bottom",
            dx=8,
            dy=-4,
            fontSize=13,
            fontWeight="bold"
        )
        .encode(
            y="media:Q",
            text=alt.value(f"Média geral = {media_geral:.2f}")
        )
    )

    return (
        grafico +
        linha_media +
        texto
    ).properties(
        title=titulo,
        width=700,
        height=400
    )

## **1. Médias por aula**

In [ ]:
graficos = {
    "nota_final": ("Nota final média por aula", [0, 10]),
    "aprendizado_sobre_conteudo": ("Aprendizado médio por aula", [1, 5]),
    "compreensao_sobre_conteudo": ("Compreensão média por aula", [1, 5]),
    "performance_professor": ("Performance média do professor por aula", [1, 5]),
    "performance_monitores": ("Performance média dos monitores por aula", [1, 5]),
}

for coluna, (titulo, dominio) in graficos.items():
    display(grafico_media_por_aula(df, coluna, titulo, dominio))

## **2. Distribuição de nota final**

In [ ]:
alt.Chart(df).mark_bar().encode(
    x=alt.X("nota_final:Q", bin=True),
    y="count()"
).properties(
    title="Distribuição da nota final"
)

## **3. BoxPlot da nota final por aula**

In [ ]:
alt.Chart(df).mark_boxplot(size=40).encode(
    x=alt.X("aula:N", title="Aula"),
    y=alt.Y(
        "nota_final:Q",
        title="Nota Final",
        scale=alt.Scale(domain=[0, 10])
    ),
    color="aula:N",
    tooltip=["aula"]
).properties(
    title="Distribuição da nota final por aula",
    width=650,
    height=400
)